# LLM Zoomcamp 2026 — Homework 04: Evaluation

This notebook presents my solution for Homework 04 of the DataTalksClub LLM Zoomcamp 2026.

The analysis is based on commit `8c1834d` of the official course repository. It covers the 72 lesson pages, 295 text chunks, and 360 ground-truth questions used in the assignment.

## Final answers

| Question | Answer |
|---|---|
| Q1 | **1400** |
| Q2 | **`01-agentic-rag/lessons/03-rag.md`** |
| Q3 | **`01-agentic-rag/lessons/01-intro.md`** |
| Q4 | **0.76** |
| Q5 | **0.55** |
| Q6 | **1** |

## Sources

### Official course material

- <https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/04-evaluation/homework.md>
- <https://github.com/DataTalksClub/llm-zoomcamp/tree/main/04-evaluation>
- <https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv>

## 1. Dependencies

In [ ]:
# Required packages
%pip install -q openai pydantic python-dotenv pandas numpy tqdm \
    minsearch gitsource onnxruntime tokenizers huggingface-hub

## 2. Supporting files and ONNX model

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

FILES = {
    "embedder.py": (
        "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/"
        "main/02-vector-search/embed/embedder.py"
    ),
    "download.py": (
        "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/"
        "main/02-vector-search/embed/download.py"
    ),
    "rag_helper.py": (
        "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/"
        "main/01-agentic-rag/code/rag_helper.py"
    ),
    "evaluation_utils.py": (
        "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/"
        "main/04-evaluation/code/evaluation_utils.py"
    ),
    "ground-truth.csv": (
        "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/"
        "main/cohorts/2026/04-evaluation/ground-truth.csv"
    ),
}

for filename, url in FILES.items():
    path = Path(filename)
    if path.exists():
        print(f"Already exists: {filename}")
    else:
        urlretrieve(url, path)
        print(f"Downloaded: {filename}")

In [ ]:
import subprocess
import sys

model_file = Path("models/Xenova/all-MiniLM-L6-v2/model.onnx")

if model_file.exists():
    print(f"Model already exists: {model_file}")
else:
    subprocess.run([sys.executable, "download.py"], check=True)

## 3. Loading the 72 official lesson pages

In [ ]:
from gitsource import GithubRepositoryDataReader

COMMIT_ID = "8c1834d"

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT_ID,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

print(f"Documents: {len(documents)}")
for document in documents[:3]:
    print(document["filename"])

assert len(documents) == 72

## 4. Q1 — Average input token count

The first question requires structured question generation for the first three lesson pages.

Using `gpt-5.4-mini`, the observed input token counts were approximately:

- page 1: `1019–1020`
- page 2: `1285–1286`
- page 3: `1752–1753`

The average is approximately `1352–1353`, so the closest option is **1400**.

In [ ]:
import json
import os
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from evaluation_utils import llm_structured

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

load_dotenv()

if os.getenv("OPENAI_API_KEY"):
    client = OpenAI()
    input_tokens = []

    for document in documents[:3]:
        prompt = json.dumps(
            {
                "filename": document["filename"],
                "content": document["content"],
            }
        )

        _, usage = llm_structured(
            client,
            data_gen_instructions,
            prompt,
            Questions,
        )

        input_tokens.append(usage.input_tokens)
        print(document["filename"], usage.input_tokens)

    average_input_tokens = sum(input_tokens) / len(input_tokens)
    q1_options = [140, 1400, 14000, 140000]
    q1_answer = min(
        q1_options,
        key=lambda option: abs(option - average_input_tokens),
    )

    print(f"Average input tokens: {average_input_tokens:.1f}")
    print(f"Q1 answer: {q1_answer}")
    assert q1_answer == 1400
else:
    print("OPENAI_API_KEY was not found, so the API call was skipped.")
    print("Reference average: approximately 1352–1353 input tokens.")
    print("Q1 answer: 1400")

### Q1: **1400**

## 5. Ground truth and chunking

In [ ]:
import pandas as pd
from gitsource import chunk_documents

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Ground-truth questions: {len(ground_truth)}")
print(f"Chunks: {len(chunks)}")
display(df_ground_truth.head())

assert len(ground_truth) == 360
assert len(chunks) == 295

## 6. Search indexes

In [ ]:
from embedder import Embedder
from minsearch import Index, VectorSearch

text_index = Index(text_fields=["content"])
text_index.fit(chunks)

def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

embedder = Embedder()
chunk_contents = [chunk["content"] for chunk in chunks]
X = embedder.encode_batch(chunk_contents)

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)
    return vector_index.search(
        query_vector=query_vector,
        num_results=num_results,
    )

print("Text and vector indexes created.")

## 7. Q2 — First text search result

In [ ]:
query = ground_truth[0]["question"]
text_result = text_search(query, num_results=1)[0]
q2_answer = text_result["filename"]

print("Question:", query)
print("Q2 answer:", q2_answer)

assert q2_answer == "01-agentic-rag/lessons/03-rag.md"

### Q2: **`01-agentic-rag/lessons/03-rag.md`**

## 8. Q3 — First vector search result

In [ ]:
vector_result = vector_search(query, num_results=1)[0]
q3_answer = vector_result["filename"]

print("Q3 answer:", q3_answer)

assert q3_answer == "01-agentic-rag/lessons/01-intro.md"

### Q3: **`01-agentic-rag/lessons/01-intro.md`**

## 9. Evaluation metrics

- **Hit Rate:** proportion of queries for which the correct document appears in the top five results.
- **MRR:** mean reciprocal rank of the first relevant result.

In [ ]:
from tqdm.auto import tqdm

def compute_relevance(record, search_function, num_results=5):
    results = search_function(
        record["question"],
        num_results=num_results,
    )
    expected = record["filename"]
    return [int(result["filename"] == expected) for result in results]

def hit_rate(relevance_total):
    hits = sum(any(relevance) for relevance in relevance_total)
    return hits / len(relevance_total)

def mrr(relevance_total):
    score = 0.0

    for relevance in relevance_total:
        for rank, is_relevant in enumerate(relevance):
            if is_relevant:
                score += 1 / (rank + 1)
                break

    return score / len(relevance_total)

def evaluate(
    search_function,
    ground_truth_data,
    num_results=5,
    show_progress=True,
):
    iterator = (
        tqdm(ground_truth_data)
        if show_progress
        else ground_truth_data
    )

    relevance_total = [
        compute_relevance(
            record,
            search_function,
            num_results=num_results,
        )
        for record in iterator
    ]

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

## 10. Q4 — Text search Hit Rate

In [ ]:
text_metrics = evaluate(text_search, ground_truth)

print(f"Hit Rate: {text_metrics['hit_rate']:.4f}")
print(f"MRR:      {text_metrics['mrr']:.4f}")

q4_options = [0.55, 0.66, 0.76, 0.88]
q4_answer = min(
    q4_options,
    key=lambda option: abs(option - text_metrics["hit_rate"]),
)

print(f"Q4 answer: {q4_answer}")
assert q4_answer == 0.76

### Q4: **0.76**

Reference value before rounding: `0.7583`.

## 11. Q5 — Vector search MRR

In [ ]:
vector_metrics = evaluate(vector_search, ground_truth)

print(f"Hit Rate: {vector_metrics['hit_rate']:.4f}")
print(f"MRR:      {vector_metrics['mrr']:.4f}")

q5_options = [0.35, 0.45, 0.55, 0.65]
q5_answer = min(
    q5_options,
    key=lambda option: abs(option - vector_metrics["mrr"]),
)

print(f"Q5 answer: {q5_answer}")
assert q5_answer == 0.55

### Q5: **0.55**

Reference value before rounding: `0.5486`.

## 12. Reciprocal Rank Fusion and hybrid search

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, document in enumerate(results):
            key = (document["filename"], document["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = document

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60, num_results=5):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)

    return rrf(
        [text_results, vector_results],
        k=k,
        num_results=num_results,
    )

## 13. Q6 — Tuning the k parameter

In [ ]:
k_values = [1, 50, 100, 200]
rows = []

for k in k_values:
    def search_with_k(query, num_results=5, _k=k):
        return hybrid_search(
            query,
            k=_k,
            num_results=num_results,
        )

    metrics = evaluate(
        search_with_k,
        ground_truth,
        show_progress=False,
    )

    rows.append(
        {
            "k": k,
            "hit_rate": metrics["hit_rate"],
            "mrr": metrics["mrr"],
        }
    )

df_hybrid = pd.DataFrame(rows)
display(
    df_hybrid.style.format(
        {"hit_rate": "{:.4f}", "mrr": "{:.4f}"}
    )
)

best_mrr = df_hybrid["mrr"].max()

# Tie-breaking rule: smallest k
q6_answer = int(
    df_hybrid.loc[df_hybrid["mrr"] == best_mrr, "k"].min()
)

print(f"Q6 answer: k = {q6_answer}")
assert q6_answer == 1

### Q6: **k = 1**

| k | Reference Hit Rate | Reference MRR |
|---:|---:|---:|
| 1 | 0.8389 | **0.6482** |
| 50 | 0.8361 | 0.6379 |
| 100 | 0.8361 | 0.6379 |
| 200 | 0.8361 | 0.6379 |

## 14. Final summary

In [ ]:
final_answers = pd.DataFrame(
    [
        {"question": "Q1", "answer": "1400"},
        {
            "question": "Q2",
            "answer": "01-agentic-rag/lessons/03-rag.md",
        },
        {
            "question": "Q3",
            "answer": "01-agentic-rag/lessons/01-intro.md",
        },
        {"question": "Q4", "answer": "0.76"},
        {"question": "Q5", "answer": "0.55"},
        {"question": "Q6", "answer": "1"},
    ]
)

display(final_answers)

expected = {
    "Q1": "1400",
    "Q2": "01-agentic-rag/lessons/03-rag.md",
    "Q3": "01-agentic-rag/lessons/01-intro.md",
    "Q4": "0.76",
    "Q5": "0.55",
    "Q6": "1",
}

actual = final_answers.set_index("question")["answer"].to_dict()
assert actual == expected
print("All six answers match the expected results.")

## Conclusion

The final answers are:

`1400`, `01-agentic-rag/lessons/03-rag.md`,
`01-agentic-rag/lessons/01-intro.md`, `0.76`, `0.55`, and `1`.

Text search reaches a Hit Rate of approximately `0.7583`. Vector search reaches an MRR of approximately `0.5486`. Among the tested hybrid-search settings, `k=1` produces the highest MRR, approximately `0.6482`.